# 06 - ResNet18 - ImageNet → COVID-QU-Syn

This notebook is **repo-script-first**: it tries to use the repository's original scripts and modules instead of reimplementing training logic inside the notebook.

## Repo usage / self-coded inventory

**Goal**: ResNet18 run for strategy **ImageNet → COVID-QU-Syn**.

**Repo-original code used**

- `scripts/train_simclr.py` for SimCLR pretraining when this strategy needs contrastive pretraining.
- `scripts/train_classification_resnet.py` for supervised ResNet classification.
- The repo's `configs/contrastive_config.yaml` and `configs/classification_config.yaml` are rewritten before the scripts are called, because those scripts read fixed config filenames.

**Still self-coded in this notebook**

- Colab setup, Google Drive mounting, dependency installation.
- Writing/patching YAML configs for the exact strategy paths.
- Running the repo scripts in order and saving console logs.

**Not self-coded here**

- No custom SimCLR training loop.
- No custom ResNet classifier training loop.

In [ ]:
# ============================================================
# Colab setup: clone repo, mount Drive, install dependencies
# ============================================================
import os, sys, subprocess, json, shutil, textwrap, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running in Colab; continuing with local paths.')

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_DIR = Path('/content/contrastive-synthesis-medcls_CVProject') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        print('Repo already exists:', REPO_DIR)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repo dir:', REPO_DIR)

# Install dependencies. Keep this lightweight; repo requirements are attempted first.
req = REPO_DIR / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=False)
if req.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'torchaudio', 'timm', 'PyYAML', 'scikit-learn',
    'seaborn', 'matplotlib', 'pandas', 'numpy', 'pillow', 'opencv-python', 'tqdm', 'wandb'
], check=False)

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU, running on CPU')

# Disable W&B prompts by default in Colab runs.
os.environ['WANDB_MODE'] = 'disabled'

# Paths
DRIVE_ROOT = Path('/content/drive/MyDrive/medcls_cvproject') if IN_COLAB else (REPO_DIR / 'local_drive_outputs')
DATA_ROOT = REPO_DIR / 'data' / 'processed'
LABELLED_DATA = DATA_ROOT / 'labelled_4232'
UNLABELLED_REAL_DATA = DATA_ROOT / 'unlabelled_16934' / 'images'
SYNTHETIC_DCGAN_DATA = DRIVE_ROOT / 'data' / 'processed' / 'synthetic_dcgan'
SYNTHETIC_ACGAN_DATA = DRIVE_ROOT / 'data' / 'processed' / 'synthetic_acgan'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs_script_first'
CONFIG_OUT = REPO_DIR / 'configs' / 'generated_script_first'
CONFIG_OUT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('LABELLED_DATA:', LABELLED_DATA)
print('UNLABELLED_REAL_DATA:', UNLABELLED_REAL_DATA)
print('SYNTHETIC_DCGAN_DATA:', SYNTHETIC_DCGAN_DATA)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

# Change to 'smoke' for quick testing.
RUN_MODE = 'full'
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

In [ ]:
# ============================================================
# Helpers: write YAML and run repo scripts with logs
# ============================================================
import subprocess, sys, yaml, os, json, time
from pathlib import Path

def write_yaml(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    print('Wrote config:', path)
    print(yaml.safe_dump(data, sort_keys=False))
    return path

def run_and_log(cmd, log_path=None):
    cmd = [str(x) for x in cmd]
    print('Running command:')
    print(' '.join(cmd))
    if log_path is None:
        subprocess.run(cmd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            f.write(line)
        ret = process.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

def count_images(root):
    root = Path(root)
    if not root.exists():
        return 0
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    return sum(1 for p in root.rglob('*') if p.suffix.lower() in exts)

def find_checkpoint(output_dir):
    output_dir = Path(output_dir)
    candidates = [
        output_dir / 'best_model.pth',
        output_dir / 'best_finetune_model.pth',
        output_dir / 'checkpoint.pth',
        output_dir / 'model.pth',
        output_dir / 'classifier.pth',
    ]
    for p in candidates:
        if p.exists():
            return p
    all_ckpts = sorted(list(output_dir.rglob('*.pth')) + list(output_dir.rglob('*.pt')), key=lambda p: p.stat().st_mtime, reverse=True)
    return all_ckpts[0] if all_ckpts else None

In [ ]:
# ============================================================
# ResNet18 strategy: ImageNet → COVID-QU-Syn
# Script-first execution order:
# 1. Optional SimCLR pretraining via scripts.train_simclr
# 2. ResNet classification via scripts.train_classification_resnet
# ============================================================
EXP_NAME = '06_resnet18_imagenet_to_covid_qu_syn'
STRATEGY = 'ImageNet → COVID-QU-Syn'
EXP_DIR = OUTPUT_ROOT / 'classification' / EXP_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if RUN_MODE == 'smoke':
    simclr_epochs = 1
    clf_epochs = 1
    batch_size = 8
else:
    simclr_epochs = 70
    clf_epochs = 70
    batch_size = 32

# Common classification data always uses real labelled data.
assert LABELLED_DATA.exists(), f'Missing labelled data: {LABELLED_DATA}'
print('Labelled count:', count_images(LABELLED_DATA))

pretrained_ckpt = ''

if True:
    PRETRAIN_DATA = SYNTHETIC_DCGAN_DATA
    assert PRETRAIN_DATA.exists(), f'Missing pretraining data: {PRETRAIN_DATA}. For synthetic runs, run notebook 00 first.'
    print('Pretraining data:', PRETRAIN_DATA, 'images:', count_images(PRETRAIN_DATA))
    simclr_save_dir = EXP_DIR / 'simclr_pretrain'
    simclr_ckpt_path = simclr_save_dir / 'simclr_backbone.pth'

    simclr_cfg = {
        # include both repo key variants because the repo config/scripts are inconsistent
        'dataset_dir': str(PRETRAIN_DATA),
        'data_dir': str(PRETRAIN_DATA),
        'image_size': 224,
        'batch_size': batch_size,
        'learning_rate': 0.001,
        'lr': 0.001,
        'weight_decay': 1e-4,
        'temperature': 0.5,
        'epochs': simclr_epochs,
        'device': device,
        'out_dim': 128,
        'num_workers': 0,
        'model_save_path': str(simclr_ckpt_path),
        'save_path': str(simclr_save_dir),
        'use_imagenet_init': True,
        'imagenet_first': True,
    }
    write_yaml(REPO_DIR / 'configs' / 'contrastive_config.yaml', simclr_cfg)
    run_and_log([sys.executable, '-m', 'scripts.train_simclr'], EXP_DIR / 'train_simclr.log')
    found = find_checkpoint(simclr_save_dir)
    pretrained_ckpt = str(found if found is not None else simclr_ckpt_path)
    print('SimCLR checkpoint selected:', pretrained_ckpt)
else:
    print('No SimCLR pretraining for this strategy.')

clf_save_path = EXP_DIR / 'resnet_classifier.pth'
clf_cfg = {
    # include both repo key variants because the repo config/scripts are inconsistent
    'dataset_dir': str(LABELLED_DATA),
    'data_dir': str(LABELLED_DATA),
    'image_size': 224,
    'batch_size': batch_size,
    'learning_rate': 1e-5 if not 'imagenet_to_covid_qu_syn' == 'none' else 1e-4,
    'lr': 1e-5 if not 'imagenet_to_covid_qu_syn' == 'none' else 1e-4,
    'weight_decay': 1e-4,
    'epochs': clf_epochs,
    'device': device,
    'num_classes': 4,
    'num_workers': 0,
    'backbone_path': pretrained_ckpt,
    'checkpoint_path': pretrained_ckpt,
    'model_save_path': str(clf_save_path),
    'save_path': str(EXP_DIR / 'finetune'),
    'output_dir': str(EXP_DIR / 'finetune'),
    'use_imagenet_init': True,
    'imagenet_first': True,
    'freeze_backbone': False,
    'strategy': STRATEGY,
}
write_yaml(REPO_DIR / 'configs' / 'classification_config.yaml', clf_cfg)
run_and_log([sys.executable, '-m', 'scripts.train_classification_resnet'], EXP_DIR / 'train_classification_resnet.log')
print('Finished:', EXP_NAME)
print('Output folder:', EXP_DIR)